In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [3]:
from google.colab import files # upload taxa abundance and saw kill mapping file here
uploaded = files.upload()

Saving combined_dataset 7.csv to combined_dataset 7.csv


In [4]:
combinedData = pd.read_csv("combined_dataset 7.csv") # need to rename this file
# for this prototype it will be a very small sample pool *unfortunately* as we are only looking at water to see how this works
waterData = combinedData[combinedData["Type"] == "Water"].copy()
print(waterData.shape)
print(waterData["Type"].value_counts())

(37, 34)
Type
Water    37
Name: count, dtype: int64


In [5]:
X = waterData[["Rain12", "Rain24", "Rain36", "Rain48", "Rain72","AirTemp", "Turbidity", "Conductivity", "DO_mgl", "salinity_ppt"]]
y = waterData[
    ["Flavobacterium", "Fluviicola", "Hydrogenophaga",
     "Hyphomicrobium", "Limnohabitans", "Nitrospira",
     "Polynucleobacter", "Pseudarcicella", "Rhodoferax",
     "Rhodoluna", "Sediminibacterium", "Sporichthyaceae",
     "TRA3-20", "Vicinamibacteraceae", "hgcI_clade"]
]
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (37, 10)
y shape: (37, 15)


In [6]:
print("Missing values in X:")
print(X.isnull().sum())
print("\nMissing values in y:")
print(y.isnull().sum())

Missing values in X:
Rain12          0
Rain24          0
Rain36          0
Rain48          0
Rain72          0
AirTemp         0
Turbidity       0
Conductivity    1
DO_mgl          1
salinity_ppt    1
dtype: int64

Missing values in y:
Flavobacterium         0
Fluviicola             0
Hydrogenophaga         0
Hyphomicrobium         0
Limnohabitans          0
Nitrospira             0
Polynucleobacter       0
Pseudarcicella         0
Rhodoferax             0
Rhodoluna              0
Sediminibacterium      0
Sporichthyaceae        0
TRA3-20                0
Vicinamibacteraceae    0
hgcI_clade             0
dtype: int64


In [7]:
# since we have missing data lets use the median fill in way
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns,
    index=X.index
)
print(X_imputed.isnull().sum())

Rain12          0
Rain24          0
Rain36          0
Rain48          0
Rain72          0
AirTemp         0
Turbidity       0
Conductivity    0
DO_mgl          0
salinity_ppt    0
dtype: int64


In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # this seed also we are using an 80 20 split
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 29
Testing samples: 8


In [9]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestRegressor(
        n_estimators=300,
        random_state=42
    )
)

In [10]:
model.fit(X_train, y_train)
print("Training complete!!")

Training complete!!


In [14]:
y_pred = model.predict(X_test)
predictions = pd.DataFrame(
    y_pred,
    columns=y.columns,
    index=y_test.index
)
print(predictions.head(8))
# predictions ran using the 8 samples that were not used to train the machine and here are all the predictions for the 15 genera

     Flavobacterium   Fluviicola  Hydrogenophaga  Hyphomicrobium  \
57       902.183333  1772.370000      303.103333        2.086667   
42       961.386667  1960.350000      289.010000        2.633333   
13       874.833333  1453.590000       86.790000        1.643333   
94       907.713333  1716.486667       66.816667        7.966667   
120      958.473333  1581.453333     1232.843333        0.880000   
84       417.130000   643.966667       15.363333        7.013333   
16       935.356667  1635.750000       87.716667        5.296667   
85       408.790000   636.243333       18.696667        7.200000   

     Limnohabitans  Nitrospira  Polynucleobacter  Pseudarcicella  Rhodoferax  \
57     2323.743333    9.120000        951.940000     3774.196667   35.243333   
42     2502.893333    6.486667       1048.980000     3860.276667   29.966667   
13     2482.693333   10.660000        640.996667     3547.206667  122.156667   
94     2660.356667   17.770000        769.006667     4206.276667   

In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
mae = mean_absolute_error(y_test, predictions, multioutput="raw_values") # how far off on average
rmse = np.sqrt(mean_squared_error(y_test, predictions, multioutput="raw_values")) # also about prediction error but weighed

results = pd.DataFrame({
    "Bacterial Taxon": y.columns,
    "MAE": mae,
    "RMSE": rmse
})
print(results)

        Bacterial Taxon          MAE         RMSE
0        Flavobacterium   419.608333   446.536150
1            Fluviicola   632.901250   677.438985
2        Hydrogenophaga   253.451667   457.349505
3        Hyphomicrobium    20.531667    31.211442
4         Limnohabitans   896.423333  1005.727912
5            Nitrospira    12.491667    13.409817
6      Polynucleobacter   251.178333   313.967568
7        Pseudarcicella  1387.668333  1533.343378
8            Rhodoferax    67.884167    79.394445
9             Rhodoluna   149.513750   171.670288
10    Sediminibacterium   479.446250   556.473331
11      Sporichthyaceae   297.995000   351.748533
12              TRA3-20    32.598333    33.646825
13  Vicinamibacteraceae     4.272083     6.466561
14           hgcI_clade   843.824167  1014.184314


Nitrospira and Vicinamibacteraceae = pretty accurate

Everything else = looking a bit rough

In [16]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test, predictions, multioutput="raw_values")
results["R2"] = r2
print(results) # Below 0 is pretty bad, 1 is best

        Bacterial Taxon          MAE         RMSE          R2
0        Flavobacterium   419.608333   446.536150   -2.953624
1            Fluviicola   632.901250   677.438985   -0.992889
2        Hydrogenophaga   253.451667   457.349505 -122.239693
3        Hyphomicrobium    20.531667    31.211442   -0.304334
4         Limnohabitans   896.423333  1005.727912   -1.088855
5            Nitrospira    12.491667    13.409817    0.237027
6      Polynucleobacter   251.178333   313.967568    0.138887
7        Pseudarcicella  1387.668333  1533.343378   -1.124605
8            Rhodoferax    67.884167    79.394445   -1.604760
9             Rhodoluna   149.513750   171.670288   -0.162005
10    Sediminibacterium   479.446250   556.473331   -0.472520
11      Sporichthyaceae   297.995000   351.748533   -0.184824
12              TRA3-20    32.598333    33.646825   -0.121802
13  Vicinamibacteraceae     4.272083     6.466561   -4.395666
14           hgcI_clade   843.824167  1014.184314    0.049891


Conclusion: The model is generating predictions and works, however, the accuracy needs a lot of improving.
I wonder if training it on the entire dataset with around 3000 columns instead of 15 may change the accuracy? Actually I don't think so because the problem here isn't the number of columns, I think it's the number of rows. We are evaluating around 10 different predictors for 15 different genera using ONLY 37 water samples, the sample number is too limited. I think what would be most beneficial for this project right now is if we find a massive dataset with 100+ samples and run the same data cleaning process again.